# Open-Meteo — Easy Global Historical Weather (Reanalysis)

**What it is.** A free, no-key weather API. The **Archive** endpoint serves daily/hourly
reanalysis history anywhere on Earth, including a ready-made **FAO reference ET (ET0)**.

| | |
|---|---|
| **Coverage** | Global, ~9 km reanalysis (ERA5 family) |
| **History** | 1940 → present (daily) |
| **Cost / key** | **Free · no key** (non-commercial; commercial needs a paid plan) |
| **Docs** | https://open-meteo.com/en/docs/historical-weather-api |

**Why it's in Tracera.** The fastest way to get clean daily weather + ET0 for **any** field,
and our global fallback where U.S.-only gridMET doesn't reach.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test — a few days of daily weather at the field

In [2]:
ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
DAILY = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum",
         "et0_fao_evapotranspiration", "shortwave_radiation_sum", "windspeed_10m_max"]

def open_meteo_daily(start, end, lat=LAT, lon=LON, daily=DAILY):
    r = requests.get(ARCHIVE, params={"latitude": lat, "longitude": lon,
        "start_date": start, "end_date": end, "daily": ",".join(daily),
        "timezone": "America/Chicago"}, timeout=60)
    r.raise_for_status()
    d = r.json()["daily"]
    return pd.DataFrame(d).assign(time=lambda x: pd.to_datetime(x["time"])).set_index("time")

test = open_meteo_daily("2023-06-01", "2023-06-05")
assert not test.empty
test

,temperature_2m_max,temperature_2m_min,precipitation_sum,et0_fao_evapotranspiration,shortwave_radiation_sum,windspeed_10m_max
time,,,,,,
2023-06-01,28.0,19.2,4.0,3.87,15.30,21.3
2023-06-02,31.1,19.5,0.0,6.07,26.20,16.3
2023-06-03,31.6,19.6,0.0,6.31,26.00,16.1
2023-06-04,30.6,19.3,0.2,6.30,26.01,19.2
2023-06-05,30.0,17.4,0.0,5.57,25.13,11.0


### 2. Core query — a full growing season + Growing Degree Days

In [3]:
wx = open_meteo_daily("2023-05-01", "2023-09-30")
tmax = wx["temperature_2m_max"].clip(upper=30)
tmin = wx["temperature_2m_min"].clip(lower=10)
wx["GDD_corn"] = (((tmax + tmin) / 2) - 10).clip(lower=0)
print(f"Season totals — precip {wx['precipitation_sum'].sum():.0f} mm | "
      f"ET0 {wx['et0_fao_evapotranspiration'].sum():.0f} mm | "
      f"GDD(corn) {wx['GDD_corn'].sum():.0f}")
wx.head()

Season totals — precip 359 mm | ET0 751 mm | GDD(corn) 1766


,temperature_2m_max,temperature_2m_min,precipitation_sum,et0_fao_evapotranspiration,shortwave_radiation_sum,windspeed_10m_max,GDD_corn
time,,,,,,,
2023-05-01,15.1,5.3,0.0,5.40,27.86,38.7,2.55
2023-05-02,15.8,4.7,0.0,5.50,28.57,32.6,2.90
2023-05-03,19.3,1.4,0.0,4.86,28.02,12.3,4.65
2023-05-04,25.8,6.2,0.0,6.52,27.78,21.3,7.90
2023-05-05,21.9,10.1,9.4,3.67,17.36,27.2,6.00


### Notes & how to extend
- **Water balance:** `precipitation_sum − et0_fao_evapotranspiration` is a quick daily
  surplus/deficit; compare against **gridMET** (US 4 km) and **OpenET** (actual ET).
- **Hourly / forecast:** swap the host for `api.open-meteo.com/v1/forecast` (16-day forecast)
  or add `hourly=...` for sub-daily detail.
- **License:** free for non-commercial use; a commercial app needs an Open-Meteo subscription
  (still inexpensive). For the US, gridMET/NASA POWER avoid that licensing entirely.